In [ ]:
import matplotlib.pyplot as plt
import matplotlib

In [ ]:
# Convert to DataFrame
df = pd.DataFrame(records_np, columns=["market_hash_name", "timestamp", "price"])
df["timestamp"] = df["timestamp"].astype(int)
df["price"] = df["price"].astype(float)//100
df["date"] = pd.to_datetime(df["timestamp"], unit="s")
df = df.sort_values(by="date")

skin_names = df["market_hash_name"].unique().tolist()
skin_names = sorted(skin_names)

output_path = "skin_list.txt"
with open(output_path, "w", encoding="utf-8") as f:
    for name in skin_names:
        f.write(name + "\n")

print(f"✅ Saved {len(skin_names)} unique skins to {output_path}")


df["price_usd"] = df["price"] 

unique_skins = ["Autograph Capsule | Counter Logic Gaming | Cologne 2016"]

plt.figure(figsize=(14, 8))
for name in unique_skins:
    subset = df[df["market_hash_name"] == name]
    plt.plot(subset["date"], subset["price_usd"], label=name)

plt.title("Skin Price History", fontsize=14)
plt.xlabel("Date")
plt.ylabel("Price (USD)")
plt.legend(
    fontsize=8,
    loc='center left',
    bbox_to_anchor=(1, 0.5),
    title="Skins"
)
plt.grid(True)
plt.tight_layout(rect=[0, 0, 0.8, 1])
plt.show()


In [ ]:
df["pct_change"] = df.groupby("market_hash_name")["price_usd"].pct_change()
df["rolling_mean"] = df.groupby("market_hash_name")["price_usd"].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean())
df["deviation_from_mean"] = (df["price_usd"] - df["rolling_mean"]) / df["rolling_mean"]
df["spike_flag"] = (
    (abs(df["pct_change"]) > 0.5)|        
    (abs(df["deviation_from_mean"]) > 0.7)
)
#Searching for the spike of a particular skin
for row in df[df["spike_flag"]].itertuples():
    if row.market_hash_name == "USP-S | Black Lotus (Factory New)":
        print(f"Spike detected for {row.market_hash_name} on {row.date.date()}:")
        print(f"  Price: ${row.price_usd:.2f}, Pct Change: {row.pct_change:.2%}, Deviation from Mean: {row.deviation_from_mean:.2%}")


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


upper_limit_usd = 500  
filtered_df = df[df["price"] < upper_limit_usd]

print(f"Found {filtered_df['market_hash_name'].nunique()} skins below ${upper_limit_usd}.")

df["market_hash_name"] = df["market_hash_name"].str.strip()

unique_skins = [
    "AK-47 | Redline (Field-Tested)",
    "SSG 08 | Ghost Crusader (Factory New)",
    "USP-S | Black Lotus (Factory New)",
    "AUG | Arctic Wolf (Factory New)",
    "AK-47 | Safari Mesh (Factory New)",
    "Sticker | Evil Geniuses (Holo) | Stockholm 2021"
]

plt.figure(figsize=(14, 8))
for name in unique_skins:
    subset = filtered_df[filtered_df["market_hash_name"] == name]
    plt.plot(subset["date"], subset["price"], label=name)

plt.title(f"Skin Price History (< ${upper_limit_usd} USD)", fontsize=14)
plt.xlabel("Date")
plt.ylabel("Price (USD)")
plt.legend(
    fontsize=8,
    loc='center left',
    bbox_to_anchor=(1, 0.5),
    title="Skins"
)
plt.grid(True)
plt.tight_layout(rect=[0, 0, 0.8, 1])
plt.show()


In [ ]:
feature_df = df.groupby("market_hash_name").agg({
    "price_usd": "mean",
    "pct_change": ["mean", "std"],
    "deviation_from_mean": "mean",
    "spike_flag": "sum"
})

feature_df.columns = ['_'.join(col).strip() for col in feature_df.columns.values]

feature_df = feature_df.rename(columns={
    "price_usd_mean": "mean_price_usd",
    "pct_change_mean": "mean_pct_change",
    "pct_change_std": "volatility",
    "deviation_from_mean_mean": "mean_deviation",
    "spike_flag_sum": "spike_count"
})